### 실습 환경 준비

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"prompt": "passage: ", "normalize_embeddings": True},
    query_encode_kwargs={"prompt": "query: ", "normalize_embeddings": True},
)

db = Chroma(
    persist_directory="./chroma_db_e5",
    collection_name="langchain-docs",
    embedding_function=embeddings,
    create_collection_if_not_exists=False,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3177.48it/s]


In [9]:
model = ChatOpenAI(model="gpt-6-luna")
retriever = db.as_retriever()

class QueryGenerationOutput(BaseModel):
    queries: list[str] = Field(..., description="검색 쿼리 목록")


prompt = ChatPromptTemplate.from_template('''
다음 문맥만을 고려해 질문에 답하세요.

문맥: """
{context}
"""

질문: {question}
''')


query_generation_prompt = ChatPromptTemplate.from_template("""\
질문에 대해 벡터 데이터베이스에서 관련 문서를 검색하기 위한
3개의 서로 다른 검색 쿼리를 생성하세요.
거리 기반 유사성 검색의 한게를 극복하기 위해
사용자의 질문에 대해 여러 관점을 제공하는 것이 목표입니다.

질문: {question}
""")


query_generation_chain = (
    query_generation_prompt
    | model.with_structured_output(QueryGenerationOutput)
    | (lambda x: x.queries)
)

### RAG-Fusion

In [7]:
# 이제 LangChain으로 RAG-Fusion을 구현해 보겠다. 먼저 RRF 처리를 함수로 구현
from langchain_core.documents import Document

def reciprocal_rank_fusion(
    retriever_outputs: list[list[Document]],
    k: int = 60
) -> list[str]:
    # 각 문서의 콘텐츠(문자열)와 그 점수의 매핑을 저장하는 딕셔너리 준비
    content_score_mapping = {}

    # 검색 쿼리마다 반복
    for docs in retriever_outputs:
        # 검색 결과의 문서마다 반복
        for rank, doc in enumerate(docs):
            content = doc.page_content

            # 처음 등장한 콘텐츠인 경우 점수를 0으로 초기화
            if content not in content_score_mapping:
                content_score_mapping[content] = 0
            # (1 / (순위 + k)) 점수를 추가
            content_score_mapping[content] += 1 / (rank + k)
    
    # 점수가 큰 순서로 정렬
    ranked = sorted(content_score_mapping.items(), key=lambda x: x[1], reverse=True)
    return [content for content, _ in ranked]

In [12]:
# 이 함수를 사용해 RAG-Fusion의 Chain 전체를 구현
rag_fusion_chain = {
    "question": RunnablePassthrough(),
    "context": query_generation_chain | retriever.map() | reciprocal_rank_fusion
} | prompt | model | StrOutputParser()

rag_fusion_chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 대규모 언어 모델(LLM)을 활용한 애플리케이션과 에이전트를 쉽게 만들 수 있도록 돕는 프레임워크입니다. 모델, 도구, 검색기 등 여러 구성 요소에 표준 인터페이스를 제공해 서로 연결하거나 바꾸기 쉽고, 외부 데이터와 LLM을 결합하는 기능도 지원합니다.\n\n간단한 에이전트 애플리케이션을 빠르게 구축할 때 유용하며, 필요에 따라 유연하게 맞춤화할 수 있습니다. LangChain은 LangGraph를 기반으로 하지만, LangChain을 사용하기 위해 LangGraph를 미리 알 필요는 없습니다.'

### Cohere 리랭크 모델 도입

In [14]:
# 여러 검색 쿼리로 검색한 결과를 Cohere의 리랭크 모델로 리랭크하는 코드는 다음과 같이 구현할 수 있다.
from typing import Any
from langchain_cohere import CohereRerank
from langchain_core.documents import Document

def rerank(inp: dict[str, Any], top_n: int = 3) -> list[Document]:
    question = inp["question"]
    retriever_outputs = inp["documents"]
    
    unique_documents = {}
    for documents in retriever_outputs:
        for doc in documents:
            if doc.page_content not in unique_documents:
                unique_documents[doc.page_content] = doc
    documents = list(unique_documents.values())

    cohere_reranker = CohereRerank(model="rerank-v5.0-pro", top_n=top_n)
    return list(cohere_reranker.compress_documents(documents=documents, query=question))


rerank_rag_chain = (
    {
        "question": RunnablePassthrough(),
        "documents": query_generation_chain | retriever.map()
    }
    | RunnablePassthrough.assign(context=rerank)
    | prompt | model | StrOutputParser()
)

rerank_rag_chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 대규모 언어 모델(LLM) 기반 애플리케이션을 개발하기 위한 프레임워크입니다. 핵심 목표는 두 가지입니다.\n\n- **다양한 모델을 쉽게 활용하도록 지원:** 제공업체마다 다른 API와 입력·출력 형식을 표준화해, 개발자가 특정 모델에 종속되지 않고 최신 모델로 바꾸기 쉽게 합니다.\n- **모델이 복잡한 작업을 수행하도록 지원:** 모델이 단순히 텍스트를 생성하는 데 그치지 않고, 도구를 동적으로 사용하고 다양한 데이터와 연산을 다루는 흐름을 구성할 수 있게 합니다. 비정형 데이터의 처리와 접근도 지원합니다.\n\nLangChain은 2022년 Python 패키지로 처음 출시됐으며, 초기에는 LLM 추상화와 RAG처럼 미리 정해진 계산 단계를 연결하는 ‘체인’을 중심으로 구성됐습니다.'